# 05. 학습 테이블: 사고(1) + 사고 없는 지점·시간(0) + 피처

입력: 03, 04 노트북 결과
출력:
- `data/3_결과/train_casecross.csv`: **주 분석용.** 사고 + 같은 지점의 평소(같은 달·같은 요일·같은 시각의 다른 주)
- `data/3_결과/train_table.csv`(전체, 검증용), `data/3_결과/train_model.csv`(무작위 대조군 버전 모델 입력, 참고용), `data/3_결과/train_table_meta.json`(무작위 대조군 확률 보정용 정보)

**주 분석: 같은 지점 비교 (case-crossover)**
사고마다 **같은 자리**에서 같은 달·같은 요일·같은 시각의 다른 주(보통 3~4개)를 "평소"(0)로 둬요. 예: 3월 14일(화) 08:20 사고 → 3월 7·21·28일(화) 08:20.
장소가 같으니 도로 모양·터널 자체 같은 장소 차이가 지워지고, A·날씨처럼 시간에 따라 변하는 것의 효과만 남아요. 결과는 "A가 오르면 사고 오즈가 몇 배"로 읽고, 확률 보정이 필요 없어요. 분석은 조건부 로지스틱 회귀(사고와 그 사고의 평소끼리만 비교)로 해요.

**참고: 무작위 대조군 (네거티브 샘플링)**
고속도로 전체 구간 × 모든 시간을 다 만들면 수억 행이라, 사고 1건마다 사고가 없었던 (지점, 시각)을 무작위로 `NEG_PER_POS`개 뽑아 0으로 둬요.
지점은 도로 길이에 비례해 고르고, 시각은 사고 데이터 기간 안에서 고릅니다. 모델이 낸 확률은 나중에 샘플링 비율로 보정해요.

**피처** (기획안 MVP 기준)
| 그룹 | 피처 |
|---|---|
| 눈부심 | `sun_elev`, `sun_azi`, `heading`, `glare_dh`(해와 주행방향 차이), `glare_25`, `glare_45`, `glare_low15`, `sun_behind` |
| 기상 | `ss` 일조, `icsr` 일사, `dc10Tca` 전운량, `vs` 시정, `ws`, `rn`, `ta`, `hm`, `dsnw` |
| 터널 | `tunnel_zone`(진입전150/경계100/이행300/기본부/출구60/진출후150/일반), `dist_to_entry_m`, `dist_from_exit_m`(터널 밖), `in_from_entry_m`, `in_to_exit_m`(터널 안), `dist_to_portal_s`(거리÷속도), `tunnel_len_m`, `tunnel_len_class`, `tunnel_axis`, `exit_sun_dh`, `exit_backlight_3`, `exit_backlight_10`, `tunnel_gap_500`, `contrast_entry`(일사×진입부) |
| 통제 | `hour`, `weekday`, `is_holiday`, `month`, `maxspeed` |
| 노출량 | 교통량: 데이터 확정 후 추가 |

도로 경사는 아직 넣지 않았어요(DEM 필요). 그래서 눈부심 플래그는 경사 0°로 보고 태양 고도만 써요.

In [ ]:
%pip install pvlib holidays geopandas scikit-learn -q

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import pvlib, holidays
from pyproj import Geod
from sklearn.neighbors import BallTree

ROOT = Path.cwd()
EXTERNAL, PROCESSED, RESULT = ROOT / "data" / "1_원본", ROOT / "data" / "2_중간", ROOT / "data" / "3_결과"
RESULT.mkdir(parents=True, exist_ok=True)
GEOD, CRS_M = Geod(ellps="WGS84"), 5179
NEG_PER_POS = 10
rng = np.random.default_rng(42)

chains = pd.read_pickle(PROCESSED / "chains.pkl").set_index("chain_id")
segments = pd.read_csv(PROCESSED / "segments.csv")
acc = pd.read_csv(PROCESSED / "accident_snapped.csv", parse_dates=["사고일시"])
weather = pd.read_csv(EXTERNAL / "asos_hourly_all.csv", parse_dates=["tm"])
stations = pd.read_csv(PROCESSED / "asos_stations_used.csv")

## 1. 사고(1), 같은 지점의 평소(0), 무작위 대조군(0) 만들기


In [ ]:
# 분 정보가 없는 사고(has_min=False)만 그 시간의 30분으로 둠 (태양 위치 계산용)
t_pos = acc.사고일시.where(acc.has_min.astype(bool), acc.사고일시 + pd.Timedelta(minutes=30))
pos = pd.DataFrame({"chain_id": acc.chain_id, "s": acc.s, "segment_id": acc.segment_id, "t": t_pos, "y": 1,
                    "case_id": np.arange(len(acc)), "design": "사고"})

n_neg = len(pos) * NEG_PER_POS
seg_pick = segments.sample(n_neg, replace=True, weights="length_m", random_state=42).reset_index(drop=True)
t0, t1 = acc.사고일시.min().floor("D"), acc.사고일시.max().ceil("D")
t_neg = t0 + pd.to_timedelta(rng.integers(0, int((t1 - t0).total_seconds() // 60), n_neg), unit="m")
neg = pd.DataFrame({"chain_id": seg_pick.chain_id, "s": rng.uniform(seg_pick.s0, seg_pick.s1),
                    "segment_id": seg_pick.segment_id, "t": t_neg, "y": 0, "case_id": -1, "design": "무작위"})

# 같은 구간·같은 시간에 실제 사고가 있었던 대조군은 제외
pos_key = set(zip(pos.segment_id, pos.t.dt.floor("h")))
neg = neg[[k not in pos_key for k in zip(neg.segment_id, neg.t.dt.floor("h"))]]

# 같은 지점의 평소 (case-crossover, time-stratified): 같은 자리, 같은 달·요일·시각의 다른 주
cc = []
for k in [-4, -3, -2, -1, 1, 2, 3, 4]:
    c = pos.copy()
    c["t"] = pos.t + pd.Timedelta(weeks=k)
    cc.append(c[c.t.dt.to_period("M") == pos.t.dt.to_period("M")])
cc = pd.concat(cc).assign(y=0, design="같은지점")
cc = cc[(cc.t >= t0) & (cc.t < t1)]
cc = cc[[k not in pos_key for k in zip(cc.segment_id, cc.t.dt.floor("h"))]]   # 그 시각에 실제 사고가 있었으면 제외

df = pd.concat([pos, neg, cc], ignore_index=True)
print(df.groupby(["design", "y"]).size())
print("사고 1건당 같은 지점 평소 개수:", cc.groupby("case_id").size().describe().round(1).to_dict())

## 2. 위치·주행 방위각

In [ ]:
def ang_diff(a, b):
    return np.abs((np.asarray(a) - np.asarray(b) + 180) % 360 - 180)

def bearing(p, q):
    g = gpd.GeoSeries([p, q], crs=CRS_M).to_crs(4326)
    return GEOD.inv(g.x[0], g.y[0], g.x[1], g.y[1])[0] % 360

pts, back, ahead = [], [], []
for c, s in zip(df.chain_id, df.s):
    line = chains.geometry[c]
    pts.append(line.interpolate(s))
    back.append(line.interpolate(max(s - 50, 0)))
    ahead.append(line.interpolate(min(s + 50, line.length)))
to_ll = lambda geoms: gpd.GeoSeries(geoms, crs=CRS_M).to_crs(4326)
p, b, a = to_ll(pts), to_ll(back), to_ll(ahead)
df["lat"], df["lon"] = p.y.to_numpy(), p.x.to_numpy()
df["heading"] = GEOD.inv(b.x.to_numpy(), b.y.to_numpy(), a.x.to_numpy(), a.y.to_numpy())[0] % 360
df["maxspeed"] = chains.maxspeed.reindex(df.chain_id).fillna(100).to_numpy()

## 3. 태양 위치와 눈부심
- `glare_dh`: 해의 방위각과 주행 방향의 차이 (0° = 해가 정면, 180° = 해가 등 뒤)
- `glare_25` / `glare_45`: 해가 낮고(고도 0~25°) 정면 ±25° / ±45° 안에 있음
- `glare_low15`: 해가 아주 낮고(0~15°) 정면 ±25°
- `sun_behind`: 해가 낮고 등 뒤(차이 135° 이상)라 앞차·표지판이 역광 없이 반사되는 상황

In [ ]:
times = pd.DatetimeIndex(df.t).tz_localize("Asia/Seoul")
sp = pvlib.solarposition.get_solarposition(times, df.lat.to_numpy(), df.lon.to_numpy())
df["sun_elev"] = sp.apparent_elevation.to_numpy()
df["sun_azi"] = sp.azimuth.to_numpy()
low = df.sun_elev.between(0, 25)
df["glare_dh"] = ang_diff(df.sun_azi, df.heading)
df["glare_25"] = low & (df.glare_dh < 25)
df["glare_45"] = low & (df.glare_dh < 45)
df["glare_low15"] = df.sun_elev.between(0, 15) & (df.glare_dh < 25)
df["sun_behind"] = low & (df.glare_dh > 135)
df.groupby("y")[["glare_25", "glare_45", "sun_behind"]].mean().round(3)

## 4. 터널 피처 (주행 방향 기준)
chain 위 위치 `s`와 터널 구간 `[입구, 출구]`를 비교해요.

| tunnel_zone | 조건 |
|---|---|
| 진입전150 | 입구 앞 150m 이내 |
| 경계100 | 터널 안, 입구에서 100m 이내 |
| 이행300 | 터널 안, 입구에서 100~400m |
| 출구60 | 터널 안, 출구까지 60m 이내 |
| 기본부 | 그 밖의 터널 안 |
| 진출후150 | 출구 뒤 150m 이내 |
| 일반 | 터널과 무관 |

In [ ]:
def tunnel_feats(c, s):
    tl = chains.tunnels[c]
    line = chains.geometry[c]
    r = dict(tunnel_zone="일반", dist_to_entry_m=np.nan, dist_from_exit_m=np.nan,
             in_from_entry_m=np.nan, in_to_exit_m=np.nan, tunnel_len_m=np.nan, tunnel_axis=np.nan, exit_bearing=np.nan, tunnel_gap_500=False)
    nxt = [t for t in tl if t[0] > s]          # 앞으로 만날 터널
    prv = [t for t in tl if t[1] < s]          # 지나온 터널
    cur = [t for t in tl if t[0] <= s <= t[1]] # 지금 터널 안
    if nxt and not cur: r["dist_to_entry_m"] = nxt[0][0] - s       # 터널 밖: 앞 터널 입구까지
    if prv and not cur: r["dist_from_exit_m"] = s - prv[-1][1]     # 터널 밖: 지나온 출구부터
    ref = cur[0] if cur else (nxt[0] if nxt and r["dist_to_entry_m"] <= 1000 else
                              prv[-1] if prv and r["dist_from_exit_m"] <= 1000 else None)   # 기준 터널
    if cur:
        d_in, d_out = s - cur[0][0], cur[0][1] - s
        r["in_from_entry_m"], r["in_to_exit_m"] = d_in, d_out
        r["tunnel_zone"] = "출구60" if d_out <= 60 else "경계100" if d_in <= 100 else "이행300" if d_in <= 400 else "기본부"
    elif nxt and r["dist_to_entry_m"] <= 150:
        r["tunnel_zone"] = "진입전150"
    elif prv and r["dist_from_exit_m"] <= 150:
        r["tunnel_zone"] = "진출후150"
    if ref:
        r["tunnel_len_m"] = ref[1] - ref[0]
        r["tunnel_axis"] = bearing(line.interpolate(ref[0]), line.interpolate(ref[1]))
        r["exit_bearing"] = bearing(line.interpolate(max(ref[1] - 50, ref[0])), line.interpolate(ref[1]))
        i = tl.index(ref)                      # 앞뒤 터널과 간격 500m 이하 = 연속 터널
        gaps = ([ref[0] - tl[i - 1][1]] if i > 0 else []) + ([tl[i + 1][0] - ref[1]] if i + 1 < len(tl) else [])
        r["tunnel_gap_500"] = any(g <= 500 for g in gaps)
    return r

tf = pd.DataFrame([tunnel_feats(c, s) for c, s in zip(df.chain_id, df.s)], index=df.index)
df = df.join(tf)
dist_portal = df[["dist_to_entry_m", "dist_from_exit_m", "in_from_entry_m", "in_to_exit_m"]].min(axis=1)
df["dist_to_portal_s"] = dist_portal / (df.maxspeed / 3.6)          # 몇 초 뒤/전에 터널 입·출구
df["tunnel_len_class"] = pd.cut(df.tunnel_len_m, [0, 500, 1000, 3000, np.inf],
                                labels=["<500", "500-1000", "1000-3000", ">3000"])
df["exit_sun_dh"] = ang_diff(df.sun_azi, df.exit_bearing)
df.loc[df.sun_elev <= 0, "exit_sun_dh"] = np.nan
df["exit_backlight_3"] = df.exit_sun_dh <= 3
df["exit_backlight_10"] = df.exit_sun_dh <= 10
pd.crosstab(df.tunnel_zone, df.y)

## 5. 기상 붙이기 (가장 가까운 관측소, ASOS `tm`은 직전 1시간 관측값이라 다음 정시에 맞춤)

In [ ]:
tree = BallTree(np.radians(stations[["stn_lat", "stn_lon"]]), metric="haversine")
dist, idx = tree.query(np.radians(df[["lat", "lon"]]), k=1)
df["stn_id"] = stations.stn_id.to_numpy()[idx[:, 0]]
df["stn_dist_km"] = dist[:, 0] * 6371                               # 지점 → 관측소 거리
df["tm"] = df.t.dt.floor("h") + pd.Timedelta(hours=1)             # 매칭 키: (관측소, 다음 정시)
df = df.merge(weather, on=["stn_id", "tm"], how="left")
df["contrast_entry"] = df.icsr * df.tunnel_zone.isin(["진입전150", "경계100"])

### 기상 매칭 근거 (검증)
기상은 **사고마다 개별로** 붙였어요. 근거는 아래 출력과 `data/3_결과/weather_match_check.csv`에 남아요.
- 매칭 키: `stn_id`(사고 지점에서 가장 가까운 ASOS 관측소) + `tm`(사고 시각을 다음 정시로 올린 값. ASOS `tm=14:00`은 13:00~14:00 관측)
- 확인할 것: 매칭률(100%에 가까워야 함), 관측소 거리 분포, 사고 몇 건의 사고시각 ↔ tm ↔ 관측소 실제 값

In [ ]:
matched = df.ta.notna()
print("매칭 키: (stn_id, tm = 사고시각의 다음 정시)")
print(f"기상 매칭률: 전체 {matched.mean():.1%} / 사고 {matched[df.y == 1].mean():.1%} / 대조군 {matched[df.y == 0].mean():.1%}")
print("관측소까지 거리(km):", df.stn_dist_km.describe(percentiles=[.5, .9]).round(1).to_dict())

# 사고 샘플: 원래 사고시각, 매칭된 관측 시각, 관측소, 거리, 실제 관측값
sample = df[df.y == 1].sample(min(10, (df.y == 1).sum()), random_state=0)[
    ["t", "tm", "stn_id", "stn_dist_km", "ta", "rn", "ss", "icsr", "dc10Tca", "vs"]]
# 원본 ASOS 파일에서 같은 (관측소, 시각) 값을 다시 찾아 붙은 값과 같은지 대조
raw = weather.set_index(["stn_id", "tm"])
sample["원본_ta"] = [raw.ta.get((s, t), np.nan) for s, t in zip(sample.stn_id, sample.tm)]
sample["일치"] = (sample.ta == sample.원본_ta) | (sample.ta.isna() & sample.원본_ta.isna())
df[df.y == 1][["t", "tm", "stn_id", "stn_dist_km", "ta", "ss", "icsr", "dc10Tca", "vs"]].to_csv(
    RESULT / "weather_match_check.csv", index=False, encoding="utf-8-sig")
sample.round(2)

## 6. 시간 통제 변수 + 저장

## 7. 터널 순응 점수 A (2팀, 기획안 "터널 순응 (2팀)" 탭)
바깥 밝기가 터널 안 밝기보다 몇 배인지(R)를 0~1 점수로 바꾼 값이에요. 클수록 눈이 적응하기 어려워요.

| 단계 | 식 | 근거 |
|---|---|---|
| E (lx) | `icsr` × 277.8 × 126 | 시간당 일사량(MJ/m²) → W/m² → 조도(lx) |
| L20 (cd/m²) | 0.15 × E / π | 운전자 시야(20°)의 바깥 휘도 |
| 조광 단계 step | clip(ceil(L20 / 4000 × 4) / 4, 0.25, 1) | 국토부 터널 조명 지침 표 6.7 조광 4단계 (25/50/75/100%) |
| L_b 경계부 휘도 | 280 × step | 280 = 4,000 × 0.07 (지침 표 6.4·6.5, 100km/h) |
| R_in | L20 / L_b | 바깥 ÷ 경계부 밝기 (진입: 블랙홀) |
| R_out | L20 / 45 | 45 = 기본부 9 × 5 (CIE, 출구부) (진출: 화이트홀) |
| A | clip(log10(R) / 2, 0, 1) | R=1 → 0, R=100 → 1 |

- `A_진입`은 tunnel_zone이 진입전150·경계100일 때만, `A_진출`은 출구60·진출후150일 때만 값이 있고 나머지는 0이에요. 밤(L20 = 0)은 A = 0이에요.
- 연속 터널 감소계수와 터널 길이는 최종 모델 변수와 겹쳐서 A에 넣지 않아요.
- 민감도 확인용 `A_진입_fixed280`(경계부 280 고정), `A_진출_307`(L20 / 30.7)은 train_table.csv에만 저장하고 모델 입력에는 넣지 않아요.

In [ ]:
E = df.icsr.fillna(0) * 277.8 * 126
L20 = 0.15 * E / np.pi
def adapt_score(R):
    R = np.asarray(R, float)
    return np.clip(np.log10(np.where(R > 0, R, 1)) / 2, 0, 1)

step = np.clip(np.ceil(L20 / 4000 * 4) / 4, 0.25, 1)      # 조광 4단계
L_b = 280 * step
entry_zone = df.tunnel_zone.isin(["진입전150", "경계100"])
exit_zone = df.tunnel_zone.isin(["출구60", "진출후150"])
df["A_진입"] = np.where(entry_zone, adapt_score(L20 / L_b), 0.0)
df["A_진출"] = np.where(exit_zone, adapt_score(L20 / 45), 0.0)
# 민감도 확인용 (train_table.csv에만 저장)
df["A_진입_fixed280"] = np.where(entry_zone, adapt_score(L20 / 280), 0.0)
df["A_진출_307"] = np.where(exit_zone, adapt_score(L20 / 30.7), 0.0)
df.groupby("tunnel_zone")[["A_진입", "A_진입_fixed280", "A_진출", "A_진출_307"]].mean().round(3)

In [ ]:
kr_holidays = holidays.KR(years=range(df.t.dt.year.min(), df.t.dt.year.max() + 1))
df["hour"] = df.t.dt.hour
df["weekday"] = df.t.dt.weekday
df["month"] = df.t.dt.month
df["is_holiday"] = df.t.dt.date.map(lambda d: d in kr_holidays) | (df.weekday >= 5)
df.to_csv(RESULT / "train_table.csv", index=False, encoding="utf-8-sig")   # 모든 중간값 포함 (검증용)

# 주 분석: 같은 지점 비교 표. 장소 변수(구간·거리·길이)는 사고와 평소가 같아서 넣지 않음
CC_COLS = ["A_진입", "A_진출", "rn", "vs", "is_holiday"]
CC_COLS += [c for c in ["log_traffic", "G"] if c in df.columns]
casecross = df[df.design.isin(["사고", "같은지점"])]
casecross = casecross[casecross.case_id.isin(casecross[casecross.y == 0].case_id)]   # 평소가 하나도 없는 사고 제외
casecross[["case_id", "segment_id", "tunnel_zone", "t", "y", *CC_COLS]].to_csv(
    RESULT / "train_casecross.csv", index=False, encoding="utf-8-sig")
print("같은 지점 비교 입력:", CC_COLS, "/ 사고", casecross.case_id.nunique(), "건")

# 모델 입력 (기획안 중복 금지 원칙): icsr·E·L20·R, contrast_entry, exit_backlight_*, exit_sun_dh,
# dist_to_portal_s, 밝기 변화속도는 A 점수의 재료이거나 눈부심 쪽과 겹쳐서 제외
MODEL_COLS = ["A_진입", "A_진출", "tunnel_zone", "dist_to_entry_m", "dist_from_exit_m", "in_from_entry_m", "in_to_exit_m",
              "tunnel_len_m", "tunnel_gap_500", "maxspeed", "hour", "weekday", "is_holiday", "rn", "vs"]
MODEL_COLS += [c for c in ["log_traffic", "G"] if c in df.columns]      # 교통량, 1팀 G는 생기면 자동 포함
rand = df[df.design.isin(["사고", "무작위"])]
rand[["segment_id", "t", "y", *MODEL_COLS]].to_csv(RESULT / "train_model.csv", index=False, encoding="utf-8-sig")
print("모델 입력:", MODEL_COLS)

# 확률 보정용 메타데이터: 대조군은 '도로 길이 × 시간' 위에서 균일하게 뽑았으므로
# 실제 사고 확률 ≈ 모델 odds × (사고 수 / 대조군 수) / (총 도로길이 km × 총 시간 h) 로 환산 가능
import json
meta = {"neg_per_pos": NEG_PER_POS, "sampling": "segment length-weighted, uniform s within segment, uniform minute in period",
        "n_pos": int((rand.y == 1).sum()), "n_neg": int((rand.y == 0).sum()),
        "network_km": float(segments.length_m.sum() / 1000), "period_start": str(t0), "period_end": str(t1),
        "period_hours": float((t1 - t0).total_seconds() / 3600), "seed": 42}
(RESULT / "train_table_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
print(df.shape, meta)
rand.groupby("y")[["sun_elev", "glare_dh", "icsr", "dc10Tca", "vs"]].mean().round(2)

### 같은 지점 비교 미리보기
A는 터널 입·출구 구간에서만 값이 있어서, A 효과에 대한 정보는 그 구간에서 난 사고에서만 나와요. 아래 표에서 그 사고 수와, 사고 순간과 평소의 A 평균을 비교해요. 사고 쪽이 평소보다 크면 "밝기 차이가 클 때 사고가 더 잦다"는 방향이에요. 본 분석(조건부 로지스틱 회귀)은 모델링 단계에서 해요.


In [ ]:
zone_cases = casecross[casecross.tunnel_zone.isin(["진입전150", "경계100", "출구60", "진출후150"])]
print("A 구간(입·출구) 사고 수:", zone_cases[zone_cases.y == 1].case_id.nunique())
zone_cases.groupby(["tunnel_zone", "y"])[["A_진입", "A_진출"]].mean().round(3).unstack("y")
